# Module 02: ITSM Queue & CMDB Bottleneck Lakehouse (PySpark + Delta Lake)
**NIST SP 800-171 Rev 2 Controls**: `3.3.1` (Audit Review), `3.4.1` (CMDB Inventory Baseline), `3.11.1` (Risk Assessment), `3.12.3` (Continuous Monitoring)  
**ITIL 4 / PMI Governance**: Incident & Service Request Handoff Optimization | Week-1 Empirical Baseline Verification (`site-mfg-01` / `enclave-cui-prod`)

This notebook ingests the raw 90-day ServiceNow exports (`incident`, `sc_req_item`, `change_request`, `sys_audit`) and CMDB CI tables (`cmdb_ci_inventory`) across Bronze, Silver, and Gold Delta layers to isolate **3+ hop ticket ping-ponging** and **orphaned CIs**.

In [ ]:
# Cell 1: Initialize SparkSession & Paths
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.appName('DefenseMfg-ITSM-Medallion-Lakehouse')
    .getOrCreate()
)

RAW_PATH = '../../data/raw'
DELTA_PATH = '../../data/delta'


In [ ]:
# Cell 2: BRONZE LAYER — Ingest Raw ServiceNow & CMDB Extracts
bronze_incidents = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/servicenow_incidents_90d.csv')
bronze_ritm = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/servicenow_sc_req_item_90d.csv')
bronze_changes = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/servicenow_change_requests_90d.csv')
bronze_sys_audit = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/servicenow_sys_audit_90d.csv')
bronze_cmdb = spark.read.option('header', 'true').option('inferSchema', 'true').csv(f'{RAW_PATH}/cmdb_ci_inventory.csv')

print(f'Ingested Incidents: {bronze_incidents.count()}, RITMs: {bronze_ritm.count()}, sys_audit hops: {bronze_sys_audit.count()}, CMDB CIs: {bronze_cmdb.count()}')


In [ ]:
# Cell 3: SILVER LAYER — Reconstruct Hop-by-Hop Ping-Ponging from sys_audit & Flag Orphaned CIs
audit_rollup = bronze_sys_audit.groupBy('documentkey').agg(
    F.count('audit_sys_id').alias('audit_hop_count'),
    F.round(F.sum('dwell_hours_before_transfer'), 2).alias('total_handoff_dwell_hours')
)

silver_incidents = (
    bronze_incidents.join(audit_rollup, bronze_incidents.number == audit_rollup.documentkey, 'left')
    .join(
        bronze_cmdb.select(
            F.col('ci_name').alias('cmdb_ci'),
            F.col('is_orphan_ci'),
            F.col('mid_discovery_status'),
            F.col('os_patch_compliant')
        ),
        on='cmdb_ci',
        how='left'
    )
    .withColumn('is_3plus_hop_ping_pong', F.col('reassignment_count') >= F.lit(3))
)

silver_incidents.filter(F.col('is_3plus_hop_ping_pong') & F.col('is_orphan_ci')).select(
    'number', 'subcategory', 'reassignment_count', 'assignment_group_path', 'cmdb_ci', 'mid_discovery_status'
).show(10, truncate=False)


In [ ]:
# Cell 4: GOLD LAYER — Rank Top 20% Ticket Subcategories Driving 80% Operational Friction
subcat_summary = silver_incidents.groupBy('category', 'subcategory', 'assignment_group_path').agg(
    F.count('number').alias('ticket_count'),
    F.round(F.avg('reassignment_count'), 2).alias('avg_hops'),
    F.round(F.sum('mttr_hours'), 2).alias('total_mttr_hours'),
    F.sum(F.when(F.col('sla_breached') == True, 1).otherwise(0)).alias('sla_breaches')
)

total_w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
pareto_w = Window.orderBy(F.col('total_mttr_hours').desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)

gold_pareto = (
    subcat_summary.withColumn('grand_total', F.sum('total_mttr_hours').over(total_w))
    .withColumn('cum_mttr', F.sum('total_mttr_hours').over(pareto_w))
    .withColumn('cumulative_friction_pct', F.round((F.col('cum_mttr') / F.col('grand_total')) * 100.0, 1))
    .drop('grand_total', 'cum_mttr')
)

gold_pareto.show(truncate=False)
